# Layer 1 â€” Data Ingestion for Guwahati Urban Prediction

This notebook pulls all raw data for the Guwahati Metropolitan Area:
1. **Sentinel-2** composites (10m, 3 epochs)
2. **Landsat 5/8** composites (30m, 5 epochs back to 2000)
3. **VIIRS** night-lights (500m, 4 years)
4. **OSM** roads & infrastructure (vector)
5. **WorldPop** population density (100m, 4 years)

All GEE exports go to **Google Drive** â†’ `guwahati_urban_prediction/` folder.

## 0. Setup & Authentication

In [ ]:
import ee

# This will pop up an OAuth dialog — click through to authorize
ee.Authenticate()
ee.Initialize(project='vivid-brand-182819')  # replace with your GEE cloud project ID

print('Earth Engine initialized:', ee.String('connected').getInfo())

## 1. Define Guwahati AOI

In [ ]:
BBOX = {'west': 91.55, 'south': 26.10, 'east': 91.88, 'north': 26.25}
aoi = ee.Geometry.Rectangle([BBOX['west'], BBOX['south'], BBOX['east'], BBOX['north']])

area_km2 = aoi.area().divide(1e6).getInfo()
print(f'Guwahati AOI area: {area_km2:.1f} kmÂ²')

## 2. Sentinel-2 â€” 10m composites (2016, 2019, 2023)

In [ ]:
def cloud_mask_s2(image):
    scl = image.select('SCL')
    clear = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    return image.updateMask(clear)

def add_indices_s2(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    return image.addBands([ndvi, ndbi, ndwi])

# Guwahati is a monsoon city — use dry season (Nov-Apr) and relax cloud filter
s2_epochs = [
    ('2016', '2016-11-01', '2017-04-30'),
    ('2019', '2019-11-01', '2020-04-30'),
    ('2023', '2023-11-01', '2024-04-30'),
]

BBOX = {'west': 91.55, 'south': 26.10, 'east': 91.88, 'north': 26.25}
aoi = ee.Geometry.Rectangle([BBOX['west'], BBOX['south'], BBOX['east'], BBOX['north']])

s2_tasks = []
for label, start, end in s2_epochs:
    col = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
        .map(cloud_mask_s2)
        .map(add_indices_s2)
    )

    count = col.size().getInfo()
    print(f'[S2] {label}: {count} scenes found (dry season, <30% cloud)')

    if count == 0:
        print(f'[S2] WARNING: No scenes for {label}, skipping export')
        continue

    bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDBI', 'NDWI']
    # Cast all bands to Float32 — normalizedDifference returns Float64, spectral bands are Float32
    composite = col.select(bands).median().clip(aoi).toFloat()

    task = ee.batch.Export.image.toDrive(
        image=composite,
        description=f'sentinel2_{label}',
        folder='guwahati_urban_prediction',
        fileNamePrefix=f'sentinel2_{label}',
        region=aoi,
        scale=10,
        crs='EPSG:4326',
        maxPixels=1e9,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True},
    )
    task.start()
    s2_tasks.append(task)
    print(f'[S2] Export started: sentinel2_{label}')

print(f'\n{len(s2_tasks)} Sentinel-2 tasks submitted.')

## 3. Landsat 5/8 â€” 30m composites (2000â€“2021)

In [ ]:
def cloud_mask_landsat(image):
    qa = image.select('QA_PIXEL')
    return image.updateMask(
        qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
    )

def scale_l8(image):
    optical = image.select(['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']).multiply(0.0000275).add(-0.2)
    return image.addBands(optical, overwrite=True)

def scale_l5(image):
    optical = image.select(['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7']).multiply(0.0000275).add(-0.2)
    return image.addBands(optical, overwrite=True)

# Dry-season windows (Nov-Apr) + relaxed cloud filter (30%)
landsat_epochs = [
    ('2000', '2000-11-01', '2001-04-30', 'LANDSAT/LT05/C02/T1_L2', scale_l5,
     ['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7'], 'SR_B4', 'SR_B3', 'SR_B5', 'SR_B2'),
    ('2005', '2005-11-01', '2006-04-30', 'LANDSAT/LT05/C02/T1_L2', scale_l5,
     ['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7'], 'SR_B4', 'SR_B3', 'SR_B5', 'SR_B2'),
    ('2010', '2010-11-01', '2011-04-30', 'LANDSAT/LT05/C02/T1_L2', scale_l5,
     ['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7'], 'SR_B4', 'SR_B3', 'SR_B5', 'SR_B2'),
    ('2016', '2016-11-01', '2017-04-30', 'LANDSAT/LC08/C02/T1_L2', scale_l8,
     ['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7'], 'SR_B5', 'SR_B4', 'SR_B6', 'SR_B3'),
    ('2021', '2021-11-01', '2022-04-30', 'LANDSAT/LC08/C02/T1_L2', scale_l8,
     ['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7'], 'SR_B5', 'SR_B4', 'SR_B6', 'SR_B3'),
]

ls_tasks = []
for label, start, end, collection, sfn, bands, nir, red, swir, green in landsat_epochs:
    col = (
        ee.ImageCollection(collection)
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUD_COVER', 30))
        .map(cloud_mask_landsat)
        .map(sfn)
    )

    count = col.size().getInfo()
    print(f'[LS] {label}: {count} scenes found (dry season, <30% cloud)')

    if count == 0:
        print(f'[LS] WARNING: No scenes for {label}, skipping')
        continue

    _nir, _red, _swir, _green = nir, red, swir, green
    def add_idx(image, _nir=_nir, _red=_red, _swir=_swir, _green=_green):
        ndvi = image.normalizedDifference([_nir, _red]).rename('NDVI')
        ndbi = image.normalizedDifference([_swir, _nir]).rename('NDBI')
        ndwi = image.normalizedDifference([_green, _nir]).rename('NDWI')
        return image.addBands([ndvi, ndbi, ndwi])

    col = col.map(add_idx)
    out_bands = bands + ['NDVI', 'NDBI', 'NDWI']
    # Cast all bands to Float32 — same fix as Sentinel-2
    composite = col.select(out_bands).median().clip(aoi).toFloat()

    task = ee.batch.Export.image.toDrive(
        image=composite,
        description=f'landsat_{label}',
        folder='guwahati_urban_prediction',
        fileNamePrefix=f'landsat_{label}',
        region=aoi,
        scale=30,
        crs='EPSG:4326',
        maxPixels=1e9,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True},
    )
    task.start()
    ls_tasks.append(task)
    print(f'[LS] Export started: landsat_{label}')

print(f'\n{len(ls_tasks)} Landsat tasks submitted.')

## 4. VIIRS Night-Lights â€” annual composites

In [ ]:
viirs_tasks = []
for year in [2016, 2019, 2021, 2023]:
    col = (
        ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')
        .filterBounds(aoi)
        .filterDate(f'{year}-01-01', f'{year}-12-31')
        .select('avg_rad')
    )
    composite = col.median().clip(aoi)

    task = ee.batch.Export.image.toDrive(
        image=composite,
        description=f'viirs_{year}',
        folder='guwahati_urban_prediction',
        fileNamePrefix=f'viirs_{year}',
        region=aoi,
        scale=500,
        crs='EPSG:4326',
        maxPixels=1e9,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True},
    )
    task.start()
    viirs_tasks.append(task)
    print(f'[VIIRS] Export started: viirs_{year}')

print(f'\n{len(viirs_tasks)} VIIRS tasks submitted.')

## 5. OSM Roads & Infrastructure\n\nOverpass API blocks requests from Colab's IP range (406 error).  \nTwo alternatives:\n- **Option A (this cell):** Use `geemap` to download OSM via the `osmnx` library\n- **Option B (recommended):** Run the Overpass download locally from your machine, then upload the GeoJSON files\n\nWe use Option A below â€” `osmnx` works from Colab without issues.

In [ ]:
!pip install -q osmnx geopandas

import osmnx as ox
import geopandas as gpd

print(f'osmnx version: {ox.__version__}')

# osmnx v2+ takes bbox as a single tuple: (west, south, east, north)
bbox = (91.55, 26.10, 91.88, 26.25)

# --- Roads (major) ---
print('[OSM] Fetching major roads...')
try:
    G_major = ox.graph_from_bbox(bbox=bbox, network_type='drive',
                                  custom_filter='["highway"~"motorway|trunk|primary|secondary"]')
    edges_major = ox.graph_to_gdfs(G_major, nodes=False)
    edges_major.to_file('osm_roads_major.geojson', driver='GeoJSON')
    print(f'[OSM] roads_major: {len(edges_major)} edges -> osm_roads_major.geojson')
except Exception as e:
    print(f'[OSM] roads_major failed: {e}')

# --- Roads (all driveable) ---
print('[OSM] Fetching all roads...')
try:
    G_all = ox.graph_from_bbox(bbox=bbox, network_type='drive')
    edges_all = ox.graph_to_gdfs(G_all, nodes=False)
    edges_all.to_file('osm_roads_all.geojson', driver='GeoJSON')
    print(f'[OSM] roads_all: {len(edges_all)} edges -> osm_roads_all.geojson')
except Exception as e:
    print(f'[OSM] roads_all failed: {e}')

# --- Railways ---
print('[OSM] Fetching railways...')
try:
    railways = ox.features_from_bbox(bbox=bbox, tags={'railway': 'rail'})
    railways.to_file('osm_railways.geojson', driver='GeoJSON')
    print(f'[OSM] railways: {len(railways)} features -> osm_railways.geojson')
except Exception as e:
    print(f'[OSM] railways failed: {e}')

# --- Infrastructure POIs ---
print('[OSM] Fetching infrastructure POIs...')
try:
    infra_tags = {
        'amenity': ['hospital', 'school', 'college', 'university', 'marketplace'],
        'office': 'government',
    }
    infra = ox.features_from_bbox(bbox=bbox, tags=infra_tags)
    infra.to_file('osm_infrastructure.geojson', driver='GeoJSON')
    print(f'[OSM] infrastructure: {len(infra)} features -> osm_infrastructure.geojson')
except Exception as e:
    print(f'[OSM] infrastructure failed: {e}')

# --- Water bodies ---
print('[OSM] Fetching water bodies...')
try:
    water_tags = {
        'waterway': ['river', 'stream', 'canal'],
        'natural': 'water',
    }
    water = ox.features_from_bbox(bbox=bbox, tags=water_tags)
    water.to_file('osm_water_bodies.geojson', driver='GeoJSON')
    print(f'[OSM] water_bodies: {len(water)} features -> osm_water_bodies.geojson')
except Exception as e:
    print(f'[OSM] water_bodies failed: {e}')

print('\n[OSM] Done. Files saved to Colab runtime â€” download them or copy to Drive.')

## 6. WorldPop â€” population density grids

In [ ]:
demo_tasks = []
for year in [2000, 2010, 2015, 2020]:
    # WorldPop UN-adjusted constrained dataset
    image = (
        ee.ImageCollection('WorldPop/GP/100m/pop_age_sex_cons_unadj')
        .filterBounds(aoi)
        .filter(ee.Filter.eq('year', year))
        .select('population')
    )

    count = image.size().getInfo()
    print(f'[WorldPop] {year}: {count} images in collection')

    if count == 0:
        # Fall back to the unconstrained version
        image = (
            ee.ImageCollection('WorldPop/GP/100m/pop')
            .filterBounds(aoi)
            .filter(ee.Filter.eq('year', year))
            .select('population')
        )
        count = image.size().getInfo()
        print(f'[WorldPop] {year}: trying unconstrained â€” {count} images')

    if count == 0:
        print(f'[WorldPop] WARNING: No data for {year}, skipping')
        continue

    mosaic = image.mosaic().clip(aoi)

    task = ee.batch.Export.image.toDrive(
        image=mosaic,
        description=f'worldpop_{year}',
        folder='guwahati_urban_prediction',
        fileNamePrefix=f'worldpop_{year}',
        region=aoi,
        scale=100,
        crs='EPSG:4326',
        maxPixels=1e9,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True},
    )
    task.start()
    demo_tasks.append(task)
    print(f'[WorldPop] Export started: worldpop_{year}')

print(f'\n{len(demo_tasks)} demographic tasks submitted.')

## 7. Check all task status

Run this cell periodically, or check https://code.earthengine.google.com/tasks

In [ ]:
# Check status and error messages for ALL tasks (not just this session's)
# This queries GEE's task list directly — works even after a kernel restart

import ee
try:
    ee.Number(1).getInfo()
except:
    ee.Authenticate()
    ee.Initialize(project='vivid-brand-182819')

print('=== All recent EE tasks ===\n')
task_list = ee.batch.Task.list()
for t in task_list[:25]:
    s = t.status()
    state = s.get('state', '?')
    desc = s.get('description', '?')
    dest = s.get('destination_uris', [])
    err = s.get('error_message', '')
    
    line = f'  {state:12s} {desc}'
    if dest:
        line += f'  -> {dest[0]}'
    if err:
        line += f'  ERROR: {err}'
    print(line)